. Teste com objetivo de determinar qual o melhor modelo com base na métrica de erro MAE;

. Primeiramente foi escolhido o teste ANOVA (Análise de Variância) de uma via, pois ele pode realizar teste de múltiplas comparações;

. Este teste verifica se existe uma diferença significativa nas médias entre as diferentes amostras;

. Este teste compara as médias MAE de todos os modelos, se a hipótese nula for rejeitada, isso indica que pelo menos um dos modelos é significativamente diferente dos outros;

. Etapas do teste ANOVA:

1. Hipóteses:
   
    . Hipótese nula (H0): Não há diferença significativa das médias MAE entre os modelos;
   
    . Hipótese alternativa (H1): Pelo menos um modelo tem uma média de MAE significativamente diferente dos outros;

3. Cálculos do Teste:

    . O teste ANOVA irá calcular a variância entre os grupos (modelos) e a variância dentro dos grupos (variação dentro de cada modelo), com base nas médias de cada grupo;

4. Resultado:

   . Se o valor-p do teste ANOVA for menor que o nível de significância 0.05, rejeitamos a hipótese nula, indicando que pelo menos um modelo tem um desempenho significativamente diferente dos outros;

In [4]:
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np

In [5]:
# MAE para cada fold de cada modelo
mae_rl_simples = [217.8948, 221.6355, 214.4085, 215.8374, 238.0413]
mae_rl_multipla = [116.9502, 112.8189, 107.9391, 104.8629, 120.2038]
mae_rl_polynomial = [29.8763, 29.1873, 28.9052, 30.8245, 31.0714]
mae_rl_svr = [37.5122, 34.5338, 41.4880, 34.4905, 36.1296]

- Estatística F: Indica a razão entre a variabilidade entre os grupos e a variabilidade dentro dos grupos;
- Quanto maior a estatística F, maior a evidência de que as médias dos grupos são diferentes;
- Valor-p: Se o valor-p for menor que 0,05, isso sugere que há uma diferença significativa nas médias de MAE entre os modelos. Caso contrário, não há evidência suficiente para rejeitar a hipótese nula.
- Neste caso o valor-p = 0.0000 sugerindo que a probabilidade de observar um valor tão extremo sob a hipótese nula é praticamente zero. Isso leva a rejeição da hipótese nula, confirmando que pelo menos um dos modelos é significativamente diferente.

In [7]:
# Realizando o teste ANOVA
f_stat, p_value = f_oneway(mae_rl_simples, mae_rl_multipla, mae_rl_polynomial, mae_rl_svr)

print(f"Estatística F: {f_stat:.4f}")
print(f"Valor-p: {p_value:.4f}")

# Interpretação do resultado
if p_value < 0.05:
    print("Rejeitamos a hipótese nula. Pelo menos um modelo é significativamente diferente.")
else:
    print("Não rejeitamos a hipótese nula. Não há diferença significativa entre os modelos.")

Estatística F: 1125.8228
Valor-p: 0.0000
Rejeitamos a hipótese nula. Pelo menos um modelo é significativamente diferente.


- O valor-p sujere diferenças significativas, deste modo, foi realizado testes post-hoc, Teste de Tukey para identificar quais modelos são significativamente diferentes entre si.
- Passos do Teste de Tukey
- Cálculo do valor-p ajustado: O teste de Tukey ajusta os valores-p para múltiplas comparações (para controlar o erro tipo I acumulado), usando uma correção chamada FWER (Family-Wise Error Rate). O valor-p ajustado ajuda a evitar falsos positivos (rejeitar a hipótese nula quando ela é verdadeira) devido ao grande número de comparações múltiplas.

In [9]:
# Agrupando os dados para o teste de Tukey
mae_values = np.concatenate([mae_rl_simples, mae_rl_multipla, mae_rl_polynomial, mae_rl_svr])
models = ['RL Simples'] * len(mae_rl_simples) + ['RL Múltipla'] * len(mae_rl_multipla) + ['RL Polinomial'] * len(mae_rl_polynomial) + ['RL SVR'] * len(mae_rl_svr)

# Realizando o teste de Tukey
tukey_result = pairwise_tukeyhsd(mae_values, models, alpha=0.05)

# Exibindo os resultados
tukey_result.summary()

group1,group2,meandiff,p-adj,lower,upper,reject
RL Múltipla,RL Polinomial,-82.582,0.0,-93.3292,-71.8348,True
RL Múltipla,RL SVR,-75.7242,0.0,-86.4714,-64.977,True
RL Múltipla,RL Simples,109.0085,0.0,98.2613,119.7557,True
RL Polinomial,RL SVR,6.8579,0.2979,-3.8893,17.6051,False
RL Polinomial,RL Simples,191.5906,0.0,180.8434,202.3378,True
RL SVR,RL Simples,184.7327,0.0,173.9855,195.4799,True
